# Husky 와이어태핑(Wire-Tapping)을 통한 부채널 파형 수집

## 두 ChipWhisperer 장치의 역할 분리 실험 — Lite (통신·프로그래머) + Husky (수동 관측자)

---

### 🎯 노트북의 목표

본 노트북은 ChipWhisperer에 처음 입문하는 동료 연구자를 대상으로, **두 대의 ChipWhisperer 장치를 동시에 운용하여 와이어태핑(wire-tapping) 방식의 부채널 파형을 수집** 하는 절차를 정리한 자료입니다.
기존 노트에서는 한 대의 장치(Husky 또는 Pro)가 *통신*, *프로그래밍*, *측정* 의 세 역할을 모두 수행했지만, 본 실험에서는 이 세 역할을 두 장치에 분배해 **실제 공격 시나리오에 더 가까운 환경** 을 구축합니다.

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **1단계** | 라이브러리 임포트 및 다중 장치(Lite + Husky) 동시 연결 | `lite_scope`, `husky_scope` |
| **2단계** | Lite ↔ 타겟 보드 통신 채널 확보 (SimpleSerial2) | `target` 객체 |
| **3단계** | 펌웨어 빌드 + Lite를 경유한 타겟 보드 프로그래밍 | 플래싱 완료된 타겟 |
| **4단계** | 골든 모델(Golden Model)을 이용한 통신·연산 정상성 검증 | 통신 검증 완료 |
| **5단계** | **Husky 와이어태핑 환경 구성** (외부 클럭 동기화 + 트리거 + ADC) | 측정 준비 완료된 Husky |
| **6단계** | `Encrypt()` + 다수 파형 수집 루프 | `t_husky`, `t_lite` 파형 배열 |
| **7단계** | Husky vs Lite 파형 비교 시각화 + 자원 해제 | 인터랙티브 Bokeh 그래프 |

> 본 자료는 SCA 입문 및 FIA 입문에 이어지는 **응용편 성격의 연구 노트** 입니다.
> SimpleSerial 패킷 구조, `my_fsr_cmd()` 헬퍼, Bokeh 시각화 등의 기본기는 가급적 반복 설명을 생략하고, **다중 장치 협업과 와이어태핑에 특화된 부분** 을 집중적으로 다룹니다.


---

## 🔍 시작하기 전에

### 와이어태핑(Wire-Tapping)이란?

전통적인 ChipWhisperer 실습에서는 단일 보드가 **타겟에게 평문을 보내고 → 연산을 트리거하고 → 전력 파형을 측정** 하는 모든 단계를 수행합니다.
즉, 측정 장비가 곧 "공격자가 통제하는 통신 단말" 입니다.

그러나 실제 부채널 공격 환경은 이렇게 협조적이지 않습니다.
공격자는 보통 **이미 동작 중인 시스템의 신호선에 측정용 프로브만을 부착** 할 수 있을 뿐, 정상 통신 흐름에는 개입할 수 없습니다.
이러한 **수동 측정(passive measurement) 시나리오** 를 실험실에서 재현하기 위해, 본 노트북은 두 대의 ChipWhisperer 장치에 서로 다른 역할을 부여합니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│  공격 시나리오 모형                                                     │
│                                                                     │
│   ChipWhisperer-Lite   ───  "정상 사용자" 역할                         │
│      └─ 타겟 보드와 UART (SimpleSerial2) 로 통신                        │
│      └─ 타겟 펌웨어를 컴파일·플래싱                                      │
│      └─ 타겟의 시스템 클럭 공급(HS2)                                    │
│                                                                     │
│   ChipWhisperer-Husky  ───  "은밀한 관측자" (wire-tap) 역할            │
│      └─ 트리거 / 클럭 / 전력 라인 세 가닥만 물리적으로 분기 측정              │
│      └─ 펌웨어·통신에는 일체 개입하지 않음                                 │
└─────────────────────────────────────────────────────────────────────┘
```

### 단일 칩위스퍼러 환경과 본 노트북 환경의 차이

| 항목 | 단일 장치 SCA | 본 노트북 (다중 장치 와이어태핑) |
|:----:|:----:|:----:|
| 측정 장비 수      | 1대 (Husky)            | **2대 (Lite + Husky)** |
| UART 통신 주체   | Husky                   | **Lite** 전담 |
| 펌웨어 프로그래머 | Husky                   | **Lite** 전담 |
| 타겟 클럭 공급원  | Husky 의 `clkgen`        | **Lite 의 `clkgen`** (HS2) |
| 측정 장치의 역할  | 통신 + 측정             | **측정만** (passive observer) |
| 측정 장치 클럭 소스 | 내부 PLL (`clkgen`)   | **외부 클럭(`extclk_aux_io`)** + 주파수 탐색 |
| 현실성 (공격 시나리오) | 낮음 (자기 자신을 측정) | **높음** (제3자가 신호선 도청) |

이러한 구성은 결과적으로 **두 가지 측정 관점** 을 동시에 확보합니다.
하나는 통신을 주관하는 Lite 의 시각이고, 다른 하나는 외부에서 신호선만 보고 있는 Husky 의 시각입니다.
이 두 파형을 나란히 시각화함으로써, 와이어태핑이 정상 측정과 얼마나 다른 신호 품질을 보여주는지 직접 비교해 볼 수 있습니다.


### 실험 환경 (물리적 배선)

```
호스트 PC (Jupyter)
    │
    │  USB ×2
    ├─────────────────────────┬──────────────────────────┐
    ▼                                                    ▼
┌──────────────────────────┐                ┌──────────────────────────┐
│   ChipWhisperer-Lite     │                │  ChipWhisperer-Husky      │
│   (active comm + power)  │                │  (passive wire-tap)       │
└──────────────────────────┘                └──────────────────────────┘
    │ 20-pin 커넥터                               │ 전면 20-pin + 측면 SMA
    │                                            │
    │  HS2  ── CLKIN  (타겟 시스템 클럭 공급)        │  D0       ←  CW308 TRIG  (CW308 GPIO4/TRIG)
    │  TX   ── RX                                │  AUX MCX  ←  CW308 CLKIN (타겟 클럭 분기, 주파수 미상)
    │  RX   ── TX                                │  Measure(Pos) ← CW308 SHUNTL (타겟 션트 양단)
    │  ...                                       │
    ▼                                            ▼
        ┌────────────────────────────────────────────┐
        │   CW308 UFO 보드 + STM32F303 (타겟 MCU)      │
        │                                            │
        │   ※ 본 실습 편의를 위해 타겟 펌웨어 내부에        │
        │      암호화 구간 진입 시 GPIO4 를 TRIG 로       │
        │      토글하도록 구현되어 있습니다.               │
        │      실제 공격에서는 통신 신호(UART idle,       │
        │      특정 패턴 등)를 트리거로 활용해야 합니다.     │
        └────────────────────────────────────────────┘
```

**연결 핵심 3선**

| 라인 | 출처 (타겟 보드) | 입력 (Husky) | 의미 |
|:----:|:----:|:----:|:----|
| 트리거 | GPIO4 / TRIG | **전면 20-pin D0** | 캡처 시점 정렬용 디지털 신호 |
| 클럭   | CLKIN         | **전면 AUX MCX**   | 타겟 동작 클럭 (정확한 주파수 미상 → 카운터로 측정) |
| 전압   | SHUNTL        | **측면 Measure (Pos)** | 션트 양단의 전압 강하 (전력 소비 비례) |

### 📖 핵심 용어집

| 용어 | 의미 |
|:----|:----|
| **와이어태핑 (Wire-Tapping)** | 통신·연산에 개입하지 않고 신호선만 분기해 측정하는 수동 관측 방식 |
| **multi-device 환경** | 호스트 PC에 두 대 이상의 ChipWhisperer가 동시 연결된 상태 |
| **`cw.list_devices()`** | USB 버스에서 ChipWhisperer 장치를 시리얼 넘버 포함해 열거하는 함수 |
| **`cw.scope(sn=...)`** | 시리얼 넘버 지정으로 특정 장치에 명시적으로 연결 |
| **`extclk_aux_io`** | PLL 입력 클럭 소스로 전면 AUX MCX 입력을 사용 |
| **`freq_ctr`** | Husky 내장 주파수 카운터의 측정값 (실시간 외부 클럭 측정) |
| **`adc_mul`** | 타겟 클럭 대비 ADC 샘플레이트 배수 (4 → 1 클럭당 4 샘플) |
| **PLL Lock** | Husky 내부 PLL이 외부 클럭에 위상·주파수 정렬 완료 상태 |
| **Golden Model** | 호스트 PC 측에서 직접 계산한 기준 출력값 (펌웨어/통신 검증용) |
| **Power Trace** | 시간에 따른 타겟의 전력 소비 측정값 (1차원 배열) |


---

# 📦 1단계 — 라이브러리 임포트 및 다중 장치 연결

> **이 단계의 목표**
> ChipWhisperer Python API와 보조 헬퍼를 로드한 뒤, 호스트 PC에 연결된 **모든** ChipWhisperer 장치를 일괄 검출해 시리얼 넘버 기반으로 두 객체(`lite_scope`, `husky_scope`)에 안전하게 할당합니다.

---

### 1.1 보조 헬퍼 로드 및 상수 정의

`My_script.ipynb` 는 본 연구 환경에서 공통으로 사용하는 헬퍼 함수와 시각화 패키지를 사전 로드합니다.

- `my_fsr_cmd(target, cmd, scmd, data, payload_only=False)` — SimpleSerial 패킷 송수신 래퍼
- `bokeh`, `numpy`, `random`, `subprocess`, `time` 등 패키지 일괄 임포트

플랫폼·프로토콜 관련 4개 상수(`PLATFORM`, `SCOPETYPE`, `CRYPTO_TARGET`, `SS_VER`)는 본 노트북 전체에서 참조되는 전역 식별자입니다.


In [1]:
# 사전 정의된 헬퍼 (my_fsr_cmd, 그래프 유틸 등)를 로드
%run My_script.ipynb

import chipwhisperer as cw

# ─────────────────────────────────────────
# 타겟 / 펌웨어 관련 상수
# ─────────────────────────────────────────
PLATFORM      = 'CW308_STM32F3'   # 타겟 보드 종류
SCOPETYPE     = 'OPENADC'         # 캡처 장치 (Lite/Husky 공통)
CRYPTO_TARGET = 'NONE'            # 사용 암호 라이브러리 (없음 -> 자체 펌웨어)
SS_VER        = 'SS_VER_2_1'      # SimpleSerial 프로토콜 버전

Loading BokehJS ...

### 1.2 호스트 PC 에 연결된 모든 ChipWhisperer 장치 검출 및 객체화

`cw.list_devices()` 는 USB 버스를 스캔해 연결된 모든 ChipWhisperer 보드를 **시리얼 넘버와 함께** 반환합니다.
시리얼 넘버를 명시해 `cw.scope(sn=...)` 로 연결하면 두 장치를 혼동 없이 분리해 다룰 수 있습니다.

`connect_all_devices()` 함수는 다음을 수행합니다:

1. 연결된 장치 목록을 받아옴
2. 장치 이름의 하이픈을 언더스코어로 치환(`ChipWhisperer-Lite` → `ChipWhisperer_Lite`)해 딕셔너리 키로 사용
3. 각 장치를 시리얼 넘버로 명시 연결 후 딕셔너리로 반환

> 💡 **시리얼 넘버 기반 연결이 중요한 이유**
> 단순히 `cw.scope()` 만 호출하면 USB 버스에서 **가장 먼저 발견된 장치 하나만** 연결됩니다.
> 두 장치를 동시에 사용하려면 반드시 시리얼 넘버를 명시해야 하며, 이는 호스트 PC가 두 장치를 안정적으로 구분하는 유일한 식별자입니다.


In [2]:
def connect_all_devices() -> dict:
    """연결된 모든 ChipWhisperer 장치에 접속하여 딕셔너리로 반환"""
    device_list = cw.list_devices()

    if not device_list:
        raise RuntimeError("연결된 ChipWhisperer 장치가 없습니다.")

    print(f"발견된 장치 수: {len(device_list)}\n")
    scopes = {}

    for device in device_list:
        # 'ChipWhisperer-Lite' → 'ChipWhisperer_Lite' 처럼 dict 키로 쓰기 쉽게 변환
        name = device['name'].replace("-", "_")
        sn   = device['sn']

        try:
            scopes[name] = cw.scope(sn=sn)
            print(f"  [✓] {name} 연결 완료  (SN: {sn})")
        except Exception as e:
            print(f"  [✗] {name} 연결 실패  (SN: {sn})\n      └─ {e}")

    return scopes

# PC에 연결된 모든 장치 인식 및 할당
scopes = connect_all_devices()
lite_scope = scopes["ChipWhisperer_Lite"]
husky_scope = scopes["ChipWhisperer_Husky"]

발견된 장치 수: 2

  [✓] ChipWhisperer_Husky 연결 완료  (SN: 502032204c5846303130313137313032)
  [✓] ChipWhisperer_Lite 연결 완료  (SN: 44203120394d36433130322030313035)


---

# 🔌 2단계 — Lite를 통한 타겟 보드 통신 채널 확보

> **이 단계의 목표**
> 타겟 보드(STM32F303)와의 **모든 시리얼 통신은 Lite가 전담** 합니다.
> SimpleSerial 프로토콜 객체를 `lite_scope` 위에 바인딩해 `target` 객체를 생성합니다.

---

본 노트북은 **SimpleSerial v2.1** 를 사용합니다.

`cw.target(lite_scope, target_type)` 호출은 다음을 수행합니다:
- Lite의 UART 핀(TX/RX)을 SimpleSerial2 송수신용으로 초기화
- 타겟과의 baud rate 협상 및 동기 바이트 확인
- 향후 모든 `target.send_cmd()` / `read_cmd()` 호출의 경로를 **Lite 경유** 로 고정

> ⚠️ **target 은 lite_scope 에만 묶인다**
> 이후 등장하는 `target.send_cmd(...)`, `my_fsr_cmd(target, ...)` 등 모든 통신은 Lite를 통해 흐릅니다.
> Husky 는 이 통신을 외부에서 **수동 관측만** 하며, 절대 통신에 개입하지 않습니다.


In [3]:
# Lite를 통한 타겟 보드 연결 설정
if SS_VER == "SS_VER_2_1":
    target_type = cw.targets.SimpleSerial2
else:
    raise OSError("지원되지 않는 SimpleSerial 버전입니다.")

try:
    target = cw.target(lite_scope, target_type)
    print("\n[✓] ChipWhisperer_Lite에 타겟 보드 연결 성공")
except Exception as e:
    print(f"\n[✗] ChipWhisperer_Lite에 타겟 보드 연결 실패: {e}")


[✓] ChipWhisperer_Lite에 타겟 보드 연결 성공


---

# 🛠 3단계 — 펌웨어 빌드 및 Lite 경유 타겟 프로그래밍

> **이 단계의 목표**
> `simpleserial_main/` 디렉터리의 펌웨어 소스를 STM32F303 용으로 컴파일하고, Lite의 SWD/JTAG 프로그래밍 기능을 이용해 타겟 보드에 플래싱합니다.

---

멀티 디바이스 환경에서는 *어느 장치가 프로그래머 역할인지* 가 중요한 셋업 정보이므로, 이를 셀에서 명시적으로 다루는 편이 이해하기 쉽기 때문입니다.

| 하위 단계 | 동작 | 비고 |
|:----:|:----|:----|
| ① 컴파일       | `make PLATFORM=... CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1` | `subprocess.run` 으로 호스트 셸 위임 |
| ② 프로그래머 선택 | `cw.programmers.STM32FProgrammer` | STM 계열 타겟용 |
| ③ Lite 기본 셋업 | `lite_scope.default_setup()` | Lite 의 클럭/UART/HS2 정상화 |
| ④ 플래싱       | `cw.program_target(lite_scope, prog, hex_path)` | **Lite 가 프로그래머 역할** |
| ⑤ 빌드 산출물 정리 | `make clean` | 작업 디렉터리 청결 유지 |

> 💡 **`lite_scope.default_setup()` 의 부수 효과**
> 이 호출은 Lite 측의 게인·ADC 샘플 수·트리거 모드를 표준값으로 초기화합니다.
> Lite 가 클럭(HS2)을 타겟에 공급하기 시작하는 시점이기도 합니다.
> 이 설정은 **Husky 측 설정과는 완전히 독립적** 이며, 5단계에서 Husky 만 따로 구성하게 됩니다.


In [4]:
# 1. 펌웨어 컴파일 (서브프로세스를 통한 bash 명령 대체)
print("펌웨어 컴파일 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 완료")

# 2. 프로그래머 설정
if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
else:
    raise OSError("프로그래머가 지원되지 않는 플랫폼입니다.")

# 3. 펌웨어 플래싱 (Lite가 프로그래머 역할을 수행)
lite_scope.default_setup()
try:
    hex_path = f"simpleserial_main/simpleserial-base-{PLATFORM}.hex"
    cw.program_target(lite_scope, prog, hex_path)
    print(f"[✓] {PLATFORM} 타겟 보드에 프로그램 업로드 완료")
except Exception as e:
    print(f"[✗] 펌웨어 프로그램 실패: {e}")

# 4. 펌웨어 컴파일 클린 (서브프로세스를 통한 bash 명령 대체)
print("펌웨어 컴파일 클린 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}", "clean"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 클린 완료")

펌웨어 컴파일 중...
펌웨어 컴파일 완료
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 100700841                 to 142697171                
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 96000000                  to 29538459                 
scope.clock.adc_rate                     changed from 96000000.0                to 29538459.0           

---

# ✅ 4단계 — 통신 검증 (Golden Model 비교)

> **이 단계의 목표**
> 본격 파형 수집에 들어가기 전에, **Lite ↔ 타겟의 통신 경로가 완벽히 살아 있는지** 골든 모델로 확인합니다.
> 통신이 불안정하면 이후 수집된 파형이 모두 무의미해지므로, 이 단계의 통과는 필수 사전 조건입니다.

---

타겟 펌웨어는 다음의 단순 연산을 수행합니다:

```c
for (i = 0; i < global_len; i++) {
    output[i] = key[i] ^ plaintext[i];
}
```

호스트에서 동일한 `k ⊕ p` 를 직접 계산한 결과가 **골든 모델** 이며, 타겟의 반환값과 바이트 단위로 일치해야 합니다.

> 💡 **이 검증이 실패하는 주요 원인**
> - SimpleSerial 버전 불일치 (펌웨어가 v1 으로 빌드되었는데 호스트는 v2 사용)
> - Lite 의 UART 핀 매핑 오류 (`default_setup()` 호출 누락)
> - 타겟의 `global_len` 미설정 (`scmd='l'` 명령 누락)
> - 펌웨어 플래싱 실패 (3단계의 `[✓]` 표시 확인)


In [5]:
MAX_DATA_LEN = 50

# 재현성을 위해 시드 고정
random.seed(1)

# 무작위 키(k), 평문(p) 생성 후 호스트에서 사전 계산한 골든 결과(k XOR p)
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
Golden_k_XOR_p = bytes(x ^ y for x, y in zip(data_k, data_p))

# 0x81 = 데이터 전송 명령 ('k'=key, 'p'=plaintext, 'l'=length)
# 0x82 = 연산 트리거 명령
# 0x83 = 결과 회수 명령
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
my_fsr_cmd(target, 0x82, 'c', [])
Return_k_XOR_p = my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

print('=== 결과 비교 ===')
print(f'타겟 결과 : {Return_k_XOR_p.hex(" ")}')
print(f'골든 모델 : {Golden_k_XOR_p.hex(" ")}')
print()
if Golden_k_XOR_p == Return_k_XOR_p:
    print('[✓] 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)')
else:
    print('[✗] 불일치! 통신 오류 또는 펌웨어 오류를 확인하세요.')

=== 결과 비교 ===
타겟 결과 : 55 d5 fe f2 29 be 4a 7d 47 d0 ce 5d 0e 60 fb eb 78 63 a9 6b 58 5d 79 02 a5 18 17 be c5 0b 74 75 74 34 19 31 88 fd 6f 55 8c 6f 8b 7e 69 61 32 7b f1 bc
골든 모델 : 55 d5 fe f2 29 be 4a 7d 47 d0 ce 5d 0e 60 fb eb 78 63 a9 6b 58 5d 79 02 a5 18 17 be c5 0b 74 75 74 34 19 31 88 fd 6f 55 8c 6f 8b 7e 69 61 32 7b f1 bc

[✓] 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)


---

# 📡 5단계 — Husky 측 와이어태핑 환경 구성

> **이 단계의 목표**
> 통신·프로그래밍·클럭 공급은 Lite 가 이미 안정적으로 수행 중인 상태에서,
> **Husky 가 통신에는 일체 개입하지 않으면서** 트리거·클럭·전력 세 신호선을 정확히 동기 측정하도록 구성합니다.
> 본 노트북의 핵심 셋업 단계이며, 이하 모든 설정은 오직 `husky_scope` 에만 적용됩니다.

---

### 5.1 와이어태핑의 클럭 동기화 전략

부채널 측정의 신호 품질은 **ADC 클럭이 타겟 클럭과 얼마나 정밀히 동기화되었느냐** 에 의해 결정됩니다 (지터 최소화).
단일 Husky 구성은 Husky가 직접 타겟에 클럭을 공급했으므로 동기화가 자동이었지만, 본 노트북에서 Husky는 **외부에서 클럭 신호를 받아오는 입장** 입니다.
즉 다음의 정보를 사전에 알지 못합니다:

- 타겟이 동작 중인 정확한 클럭 주파수
- 클럭의 위상

따라서 Husky는 아래의 단계로 외부 클럭을 **탐색·정렬** 해야 합니다.

```
[1] PLL 입력 소스를 외부 AUX 로 전환                  ← 본 단계
[2] AUX MCX 핀을 high-Z 입력 모드로 설정             ← 본 단계
[3] 내장 주파수 카운터로 외부 클럭 주파수 측정       ← 다음 단계
[4] 측정된 주파수로 PLL의 목표 주파수 설정           ← 다음 단계
[5] ADC 클럭을 타겟 클럭의 4배로 정렬 (adc_mul=4)    ← 다음 단계
[6] ADC 리셋 후 lock 상태 확인                       ← 다음 단계
```

먼저 PLL 입력 소스와 AUX 핀 모드를 설정합니다.


In [6]:
# 이전 설정의 간섭을 방지하기 위해 Husky 공장 초기화
husky_scope.default_setup()
time.sleep(0.5)

print("Husky 스코프 하드웨어 동기화 및 튜닝 중...")

# ---------------------------------------------------------
# [1] 클럭 신호 탐색 및 동기화 (Aux in/out)
# ---------------------------------------------------------
husky_scope.clock.clkgen_freq = 0
husky_scope.clock.reset_adc()
# AUX MCX 를 입력(high-Z)으로 설정 → Husky 가 클럭을 driving 하지 않고 수신만 함
husky_scope.io.aux_io_mcx = 'high_z'
# PLL 입력 소스를 외부 클럭(extclk)으로 지정
husky_scope.clock.clkgen_src = 'extclk_aux_io'
husky_scope.clock.reset_adc()

if (husky_scope.io.aux_io_mcx == 'high_z') and (husky_scope.clock.clkgen_src == 'extclk_aux_io'):
    print(f"[✓] io.aux_io_mcx     = {husky_scope.io.aux_io_mcx}")
    print(f"[✓] clock.clkgen_src  = {husky_scope.clock.clkgen_src}")
else:
    print(f"[✗] 외부 클럭 설정 실패")

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.io.cdc_settin

### 5.2 외부 클럭 주파수 탐색 + ADC 4× 오버샘플링 정렬

핵심 한 줄은 다음입니다:

```python
husky_scope.clock.clkgen_freq = husky_scope.clock.freq_ctr
```

`freq_ctr` 는 Husky 내장 주파수 카운터가 실시간으로 측정한 외부 클럭 값입니다.
이 값을 PLL의 목표 주파수로 그대로 대입해 **"내가 본 외부 클럭에 맞춰 내부 PLL을 잠근다"** 는 의미가 됩니다.
이어서 `adc_mul = 4` 로 ADC 샘플레이트를 타겟 클럭의 4배(약 30 MHz)로 설정해 클럭당 4 샘플의 정밀도를 확보합니다.

| 설정 | 값 | 의미 |
|:----:|:----:|:----|
| `freq_ctr_src` | `'extclk'` | 카운터의 측정 대상으로 외부 클럭 지정 |
| `clkgen_freq`  | `freq_ctr` (≈ 7.4 MHz) | 측정된 주파수에 PLL 목표 잠금 |
| `adc_mul`      | `4` | 1 클럭 → 4 ADC 샘플 (SCA용 미세 누설 분석에 적합) |

In [7]:
# ---------------------------------------------------------
# [2] 외부 클럭 주파수 탐색 및 동기화
# ---------------------------------------------------------
# 더 정밀한 클럭 주파수 설정 가능 (동기화 및 PLL 잠금이 실패할 경우에 주석처리하면 동작할 수 있습니다)
husky_scope.clock.pll._allow_rdiv = True
# 주파수 카운터의 측정 대상을 외부 클럭으로 지정
husky_scope.clock.freq_ctr_src = 'extclk'
# 카운터 초기 안정화를 위한 짧은 대기
time.sleep(0.5)

# 주파수 데이터 수집 (0.2초 간격, 20회)
freqs = []
for _ in range(20):
    freqs.append(husky_scope.clock.freq_ctr)
    time.sleep(0.2)
data = pd.Series(freqs)
print(f"최빈값: {data.mode().iloc[0]} (등장 {(data == data.mode().iloc[0]).sum()}/{len(data)}회)")
print(f"범위: {data.min()} ~ {data.max()} (Δ={data.max()-data.min()})")
print("\n[전체 통계 요약]")
print(data.describe()) # 개수, 평균, 표준편차, 최소, 최대, 사분위수 출력

# 내부/외부 클럭 주파수 동기화
husky_scope.clock.clkgen_freq = data.mode().iloc[0]
# ADC 샘플레이트 = 타겟 클럭 × 4 (4× 오버샘플링)
husky_scope.clock.adc_mul = 4
# ADC 리셋
husky_scope.clock.reset_adc()

if husky_scope.clock.adc_locked:
    print("[✓] ADC 클럭 동기화 완료")
    print(f"   - ADC 샘플레이트 (adc_freq): {husky_scope.clock.adc_freq:,.0f} Hz")
else:
    print("[✗] ADC 클럭 동기화 실패 (Lock Error)")

if husky_scope.clock.clkgen_locked:
    print("[✓] Husky PLL 잠금 성공")
    print(f"   - 타겟 클럭 (clkgen_freq) : {husky_scope.clock.clkgen_freq:,.0f} Hz")
else:
    print("[✗] Husky PLL 잠금 실패! 외부 클럭의 진폭/듀티/안정성을 확인하세요.")

최빈값: 7384483 (등장 16/20회)
범위: 7384483 ~ 7384494 (Δ=11)

[전체 통계 요약]
count    2.000000e+01
mean     7.384485e+06
std      4.514305e+00
min      7.384483e+06
25%      7.384483e+06
50%      7.384483e+06
75%      7.384483e+06
max      7.384494e+06
dtype: float64


(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:585) scope.clock.pll._allow_rdiv is True; this can cause an inconsistant phase relationship between the target and sampling clocks. Do you really want this?
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:585) scope.clock.pll._allow_rdiv is True; this can cause an inconsistant phase relationship between the target and sampling clocks. Do you really want this?


[✓] ADC 클럭 동기화 완료
   - ADC 샘플레이트 (adc_freq): 29,537,932 Hz
[✓] Husky PLL 잠금 성공
   - 타겟 클럭 (clkgen_freq) : 7,384,483 Hz


> `husky_scope.clock.pll._allow_rdiv = True`는 더 정밀한 추적을 가능하게 합니다.
>> ⚠️ **PLL이 정확한 주파수에 잠기지 않을 수 있음**
>> `husky_scope.clock.pll._allow_rdiv = True`을 주석처리하면, 위 셀의 출력에서 `Could not calculate pll settings for the requested frequency (7384506); generating a 7400000 clock instead.` 와 같은 메시지가 보일 수 있습니다.
>> Husky의 PLL이 임의의 분수 주파수를 정확히 합성하지 못해 **가장 가까운 합성 가능 주파수** 로 대체한다는 의미입니다.


### 5.3 트리거 입력 핀 설정

타겟 펌웨어는 암호화 진입 시 GPIO4 라인을 LOW → HIGH 로 토글합니다.
이 신호를 Husky 전면 20-pin의 **D0** 으로 받아 들여, **상승 엣지(rising edge)** 에 캡처를 시작하도록 설정합니다.

| 설정 | 값 | 의미 |
|:----:|:----:|:----|
| `trigger.triggers` | `'userio_d0'` | 전면 USERIO D0 핀을 트리거 입력으로 사용 |
| `trigger.module`   | `'basic'`     | 단순 엣지/레벨 검출 모듈 |
| `adc.basic_mode`   | `'rising_edge'` | 상승 엣지에서 캡처 시작 |


In [8]:
# ---------------------------------------------------------
# [3] 트리거 핀 설정 (전면 USERIO - D0 핀)
# ---------------------------------------------------------
# 트리거 입력 소스 = 전면 USERIO D0
husky_scope.trigger.triggers = 'userio_d0'
# 단순 엣지/레벨 검출용 'basic' 트리거 모듈 사용
husky_scope.trigger.module = 'basic'
# 캡처 시작 조건 = 상승 엣지 (타겟이 트리거를 LOW → HIGH 로 토글)
husky_scope.adc.basic_mode = 'rising_edge'

print("[✓] Husky 스코프 파라미터 설정 완료")
print(f"trigger.triggers = {husky_scope.trigger.triggers}")
print(f"trigger.module   = {husky_scope.trigger.module}")
print(f"adc.basic_mode   = {husky_scope.adc.basic_mode}")

[✓] Husky 스코프 파라미터 설정 완료
trigger.triggers = userio_d0
trigger.module   = basic
adc.basic_mode   = rising_edge


### 5.4 ADC 캡처 파라미터 (게인 / 샘플 수 / 오프셋)

션트 양단의 차동 입력은 측면 **Measure (Pos/Neg)** 핀을 통해 곧바로 Husky 내부 LNA → ADC 경로로 흘러갑니다.
별도 라우팅 설정 없이 **게인** 과 **샘플 수** 만 적절히 조절하면 됩니다.

| 파라미터 | 값 | 설명 |
|:----:|:----:|:----|
| `gain.db`        | `25` (dB) | LNA 게인. 너무 높으면 클리핑, 너무 낮으면 SNR 저하 |
| `adc.samples`    | `10000`   | 한 번의 캡처에서 수집할 샘플 개수 |
| `adc.offset`     | `0`       | 트리거 후 캡처 시작점 (0 = 트리거 즉시 시작) |
| `adc.presamples` | `0`       | 트리거 직전에 추가로 수집할 샘플 수 |

> 💡 **`gain.db = 25` 의 의미**
> 입력값은 사용자가 25 를 요청하지만, Husky의 게인 단계가 이산값(quantized)이므로 실제로는 약 `25.09` 처럼 가까운 값에 맞춰집니다.
> 이는 정상이며, 출력 메시지로 실제 적용된 값을 확인할 수 있습니다.

> 🔬 **`adc.samples` 산정 가이드**
> 본 실습 펌웨어의 XOR 연산은 적은 샘플 내에 완료됩니다. 와이어태핑 시 파형 형태의 비교를 위해 `10000` 으로 설정했습니다.

In [9]:
# LNA 게인 (dB). 보통 20~30 dB 사이. 너무 높으면 클리핑, 너무 낮으면 SNR 저하.
husky_scope.gain.db = 25

# 한 번의 캡처에서 수집할 샘플 개수
husky_scope.adc.samples = 10000

# 트리거 이후 캡처 시작점 (0 = 트리거 즉시 캡처 시작)
husky_scope.adc.offset = 0

# 트리거 이전 샘플 (사전 캡처). 필요 시 양수로 설정 가능.
husky_scope.adc.presamples = 0

print(f"gain.db          = {husky_scope.gain.db}")
print(f"adc.samples      = {husky_scope.adc.samples}")
print(f"adc.offset       = {husky_scope.adc.offset}")
print(f"adc.presamples   = {husky_scope.adc.presamples}")

gain.db          = 25.091743119266056
adc.samples      = 10000
adc.offset       = 0
adc.presamples   = 0


### 5.5 (비교 분석 옵션) Lite 스코프의 평행 설정

본 노트북은 **동일 연산에 대한 Lite 측 파형과 Husky 측 파형을 동시 캡처해 비교** 합니다.
- Lite : 통신·프로그래밍을 담당하면서 *내장 ADC* 로 션트 라인을 직접 측정 (편의상 가능)
- Husky : 외부 와이어태핑으로 동일 라인 측정

두 파형이 형태적으로 일치하면 **와이어태핑 경로의 신호 품질이 정상 측정에 준한다** 는 증거가 됩니다.
공정 비교를 위해 두 스코프의 핵심 캡처 파라미터(게인, 샘플 수, 오프셋, ADC 배수)를 **동일하게** 정렬합니다.

> 💡 **Lite 측 클럭은 외부 탐색이 필요 없다**
> Lite 가 HS2 로 타겟에 클럭을 *공급하는 주체* 이기 때문에, 자기 자신의 PLL은 이미 동일 주파수로 잠겨 있습니다.
> 따라서 `clkgen_freq` 재설정 절차가 불필요하며 `adc_mul` 만 정렬하면 충분합니다.


In [10]:
# 비교용 (cw-lite)
lite_scope.clock.adc_mul = 4
lite_scope.clock.reset_adc()
lite_scope.gain.db = 25
lite_scope.adc.samples = 10000
lite_scope.adc.offset = 0
lite_scope.adc.presamples = 0

print(f"gain.db          = {lite_scope.gain.db}")
print(f"adc.samples      = {lite_scope.adc.samples}")
print(f"adc.offset       = {lite_scope.adc.offset}")
print(f"adc.presamples   = {lite_scope.adc.presamples}")

gain.db          = 24.8359375
adc.samples      = 10000
adc.offset       = 0
adc.presamples   = 0


---

# 🌊 6단계 — `Encrypt()` 함수 정의 및 다수 파형 수집 루프

> **이 단계의 목표**
> 1회의 암호화 트랜잭션(키·평문 주입 → 길이 설정 → 트리거 → 결과 회수)을 단일 함수 `Encrypt()` 로 구현하고,
> 두 스코프(`lite_scope`, `husky_scope`)를 **동시에 arm** 한 상태에서 N회 반복 캡처합니다.

---

### 6.1 `Encrypt()` 함수의 역할

```python
Encrypt(data_k, data_p) → bytes  # 반환: 타겟이 계산한 k ⊕ p
```

내부는 4단계로 구성됩니다.

| 단계 | 명령 | 의미 |
|:----:|:----:|:----|
| ① | `0x81 'k'` | 키 주입 |
| ② | `0x81 'p'` | 평문 주입 |
| ③ | `0x81 'l'` | 출력 길이(`MAX_DATA_LEN`) 통보 |
| ④ | `0x82 'c'` | 연산 트리거 (펌웨어가 GPIO4 토글 → Husky 캡처 시작) |
| ⑤ | `0x83 'r'` | 결과 회수 |

### 6.2 수집 루프의 동시 arm 패턴

다중 스코프 환경의 핵심 패턴은 다음의 순서를 **반드시** 지키는 것입니다:

```
1. husky_scope.arm()      ─┐
2. lite_scope.arm()        │  ← 두 스코프 모두 "트리거 대기" 상태로 진입
3. Encrypt(...)            │  ← Lite 가 타겟과 통신하면 타겟이 GPIO4 트리거 발생
4. husky_scope.capture()   │  ← 두 스코프가 동일 트리거에 동시 반응
5. lite_scope.capture()   ─┘
6. get_last_trace() ×2
```

> ⚠️ **arm 순서를 뒤바꾸지 말 것**
> `Encrypt()` 를 호출한 *후* 에 `arm()` 을 부르면 트리거가 이미 지나가 버려 캡처가 실패합니다.
> 두 스코프 모두 arm된 상태에서 트리거가 한 번 발생해야 두 파형이 **시간축으로 정렬** 됩니다.

> 💡 **재현성을 위한 시드 고정**
> `random.seed(1)` 로 동일한 (key, plaintext) 시퀀스를 보장합니다.
> 이후 분석 단계(CPA 등)에서 동일 입력 분포에 대해 재계산할 수 있어 디버깅이 쉬워집니다.


In [11]:
def Encrypt(data_k, data_p):    
    my_fsr_cmd(target, 0x81, 'k', data_k)
    my_fsr_cmd(target, 0x81, 'p', data_p)
    my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
    my_fsr_cmd(target, 0x82, 'c', [])  
    return my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

# 수집 데이터 초기화
t_husky = []
t_lite = []
i_k = []
i_p = []
o_c = []

N_TRACES = 10 # 수집할 총 파형 개수
print(f"\n=== [{N_TRACES}]개의 파형 수집을 시작합니다 ===")

# 재현성을 위해 시드 고정
MAX_DATA_LEN = 50  # 한 번에 전송 가능한 최대 데이터 크기 (바이트)
random.seed(1)

for i in range(N_TRACES):
    # 1. 랜덤 생성
    data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
    data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
    
    # 2. Husky 스코프를 Arm 상태로 전환하여 D0 핀의 트리거 신호 대기
    husky_scope.arm()
    lite_scope.arm()
    
    # 3. Lite 스코프를 통해 타겟 보드로 통신 (이 과정에서 타겟이 트리거 발생)
    ct = Encrypt(data_k, data_p)
    
    # 4. Husky가 트리거를 성공적으로 인식했는지 타임아웃 여부 점검
    ret_husky = husky_scope.capture()
    ret_lite = lite_scope.capture()
    if ret_husky or ret_lite:
        print(f"  [✗] 파형 {i} 수집 실패: 타임아웃 발생! 타겟 펌웨어 혹은 D0 핀 연결 상태를 확인하세요.")
        continue
        
    # 5. 수집된 파형 데이터 추출 및 리스트 저장
    wave_husky = husky_scope.get_last_trace()
    wave_lite = lite_scope.get_last_trace()
    
    t_husky.append(wave_husky)
    t_lite.append(wave_lite)
    i_k.append(data_k)
    i_p.append(data_p)
    o_c.append(ct)
    
    # 10개 단위로 진행 상황 모니터링
    if (i + 1) % 10 == 0:
        print(f"  - 진행 상황: {i + 1} / {N_TRACES} 캡처 완료")

print("\n[✓] 모든 파형 수집이 성공적으로 완료되었습니다!")


=== [10]개의 파형 수집을 시작합니다 ===
  - 진행 상황: 10 / 10 캡처 완료

[✓] 모든 파형 수집이 성공적으로 완료되었습니다!


---

# 📊 7단계 — 수집 결과 시각화 (Husky vs Lite 비교)

> **이 단계의 목표**
> 동일 연산에 대해 두 스코프가 캡처한 파형을 **인터랙티브 Bokeh 그래프로 겹쳐 그려** 신호 품질과 와이어태핑 경로의 충실도를 시각적으로 검증합니다.

---

### 7.1 Husky (와이어태핑) 파형

Husky 가 외부에서 션트·클럭·트리거 세 가닥으로 관측한 결과입니다.
N=10 개의 파형을 겹쳐 그려 **수집 재현성** 을 점검합니다.

> 💡 **Bokeh 인터랙션 활용 팁**
> - 마우스 휠 → 시간축 확대/축소
> - 박스 줌 → 관심 구간 클로즈업
> - 범례 클릭 → 해당 trace 만 숨김
> - Hover → (sample index, 전력값) 정확히 확인


In [12]:
plot_t(t_husky, num_plot=10)
print('\n[✓] 시각화 완료!')


[✓] 시각화 완료!


### 7.2 Lite (정상 측정) 파형 — 비교용

Lite 가 자기 자신의 ADC 로 측정한 결과입니다.
**같은 시간 구간에 같은 연산** 을 보고 있으므로, 와이어태핑 경로가 신호를 충실히 전달하고 있다면 두 그래프의 패턴은 거의 동일해야 합니다.


In [13]:
plot_t(t_lite, num_plot=10)
print('\n[✓] 시각화 완료!')


[✓] 시각화 완료!


> 🔬 **두 파형을 비교하며 점검할 사항**
>
> - **거시 패턴 일치 여부** — 두 그래프의 전반적인 형상(피크 위치, 반복 패턴)이 일치하는가?
> - **상대 진폭** — Husky 와 Lite 의 측정 진폭은 LNA 게인·임피던스 매칭에 따라 다를 수 있으나, **상대적인 변동 구조** 는 같아야 합니다.
> - **노이즈 플로어** — 와이어태핑 경로는 케이블이 길어 노이즈가 더 클 수 있습니다. SNR 저하가 심하다면 케이블을 짧게 하거나 게인을 재조정하세요.
> - **시간축 정렬** — 두 그래프에서 동일한 특징점(예: 첫 번째 큰 피크)의 sample index 가 거의 같아야 합니다. 어긋난다면 트리거 지터·PLL 안정성을 점검합니다.
>
> 두 파형이 거시적으로 일치한다면, 이후 단계에서는 **Husky 데이터만으로도** CPA·DPA 등 본격 부채널 공격 절차를 수행할 수 있다는 의미입니다.
> 이는 곧 **현실적 와이어태핑 공격 시나리오의 실험적 타당성** 을 확보한 것이기도 합니다.


---

# 🔚 마무리 — 다중 장치 자원 해제

> **이 단계의 목표**
> 노트북 종료 전에 **두 스코프와 타겟 객체를 모두 명시적으로 해제** 해 다음 세션의 USB 점유 충돌을 방지합니다.

---

해제 순서는 다음을 반드시 지킵니다:

1. **`target.dis()`** — UART 채널 (Lite 가 점유 중) 해제
2. **각 scope.dis()** — Husky, Lite 순으로 USB 디바이스 핸들 해제

> ⚠️ **타겟을 먼저 해제하지 않으면 발생하는 문제**
> `target` 객체는 Lite 의 UART 핀을 점유하고 있습니다.
> 이를 닫지 않고 `lite_scope.dis()` 를 먼저 호출하면 차회 실행 시 UART 핀이 점유된 채로 남아 `target` 재생성이 실패할 수 있습니다.


In [14]:
def disconnect_all_devices(scopes: dict) -> None:
    # 타겟 객체 먼저 닫기 (Lite 의 UART 점유 해제)
    try:
        target.dis()
        print("  [✓] 타겟 보드 (SimpleSerial2) 연결 해제 완료")
    except Exception as e:
        print(f"  [✗] 타겟 보드 연결 해제 실패  └─ {e}")

    """딕셔너리 내 모든 장치 연결 해제"""
    print("\n장치 연결 해제 중...")
    for name, scope in scopes.items():
        try:
            scope.dis()
            print(f"  [✓] {name} 연결 해제 완료")
        except Exception as e:
            print(f"  [✗] {name} 연결 해제 실패\n      └─ {e}")
    scopes.clear()

# 파형 수집 및 데이터 저장이 끝난 후 반드시 포트 및 메모리 자원 반환
disconnect_all_devices(scopes)

  [✓] 타겟 보드 (SimpleSerial2) 연결 해제 완료

장치 연결 해제 중...
  [✓] ChipWhisperer_Husky 연결 해제 완료
  [✓] ChipWhisperer_Lite 연결 해제 완료


---

## 📝 본 노트북 요약

| 단계 | 핵심 함수 / 명령 | 결과 |
|:----:|:---|:---|
| 1 | `cw.list_devices()` + `cw.scope(sn=...)` | 다중 장치 동시 연결 (Lite + Husky) |
| 2 | `cw.target(lite_scope, SimpleSerial2)`     | Lite ↔ 타겟 통신 채널 확립 |
| 3 | `make` + `cw.program_target(lite_scope, ...)` | Lite 가 프로그래머로 동작 |
| 4 | `my_fsr_cmd()` + Golden Model 비교         | 통신·연산 정상성 검증 |
| 5 | `clkgen_src='extclk_aux_io'` + `freq_ctr` 매칭 | Husky 가 외부 클럭에 PLL 잠금 |
| 6 | `husky.arm()` + `lite.arm()` → `Encrypt()` → 양쪽 `capture()` | 두 스코프 동시 캡처 |
| 7 | Bokeh overlay (Husky vs Lite)              | 와이어태핑 신호 품질 시각 검증 |

### ✅ 본 노트북에서 익혀야 할 핵심 개념

1. **장치 역할 분리** — 통신·프로그래밍은 Lite, 측정은 Husky 가 전담하는 다중 장치 협업 구조
2. **시리얼 넘버 기반 명시 연결** — `cw.list_devices()` → `cw.scope(sn=...)` 패턴으로 두 장치를 안전하게 구분
3. **외부 클럭 동기화** — `extclk_aux_io` 입력 + `freq_ctr` 측정 + PLL 잠금의 3-스텝 부트스트랩
4. **동시 arm 패턴** — 두 스코프를 모두 arm 한 뒤 단일 트리거로 정렬 캡처
5. **와이어태핑 충실도 검증** — Husky vs Lite 파형 비교로 측정 경로의 신뢰성 확인

---
*Husky 와이어태핑 부채널 파형 수집 — 응용 노트북 끝*
